In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, IFrame, Markdown, display
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 170)
RES = ROOT / "results"
def show_png(path, width=620):
    display(Image(filename=str(path), width=width))

# 06 · API and UI integration
The API (`api/main.py`) loads the model, feature list and decoder config **once** at start-up. `POST /predict` runs the prediction
(features → classifier → Viterbi) and only **afterwards** `api/evaluation.py` reads the held-out route and returns two numbers.
This notebook calls the API in-process (FastAPI `TestClient`) in rehearsal mode on the student_test trips (`PATHSNAP_EVAL_SPLIT=student_test`);
the same code serves `eval_test` at evaluation time.

In [2]:
os.environ["PATHSNAP_EVAL_SPLIT"] = "student_test"
import io, contextlib
from fastapi.testclient import TestClient
from api.main import app
buf = io.StringIO()
with contextlib.redirect_stdout(buf), TestClient(app) as client:
    health = client.get("/health").json()
    trips = client.get("/trips").json()
    first = client.post("/predict", json={"trip_id": trips[0]}).json()
    rows = []
    for t in trips:
        r = client.post("/predict", json={"trip_id": t}).json()
        rows.append({"trip": t, "n_points": r["n_points"], "accuracy": r["accuracy"], "route_continuity": r["route_continuity"], "latency_ms": r["latency_ms"]})
print(health, trips)
print("response fields:", sorted(first))
print("first points:", first["points"][:2])
print("model version:", first["model_version"])

{'status': 'ok'} ['00000000', '00000007', '00000009', '00000040', '00000052', '00000056', '00000065', '00000076', '00000077', '00000091']
response fields: ['accuracy', 'latency_ms', 'model_version', 'n_points', 'points', 'route_continuity', 'route_coords', 'trip_id']
first points: [{'point_idx': 0, 'lon': 54.963519, 'lat': 56.251559, 'arc_id': 17630}, {'point_idx': 1, 'lon': 54.963143, 'lat': 56.251523, 'arc_id': 17630}]
model version: 68d8c740


## Prediction for every served trip (each is one `/predict` call)

In [3]:
df = pd.DataFrame(rows).round(4)
display(df)
print("one arc per GPS point on every trip:", all(r["n_points"] > 0 for r in rows))

,trip,n_points,accuracy,route_continuity,latency_ms
0,00000000,2503,0.9680,0.9968,677.0195
1,00000007,1354,0.9941,0.9993,406.7451
2,00000009,1049,0.9142,0.9532,403.5426
3,00000040,2995,0.9800,0.9963,848.9144
4,00000052,262,0.9885,0.9693,105.1541
5,00000056,3958,0.9593,0.9712,1190.2418
6,00000065,768,0.8763,0.9804,369.1631
7,00000076,1486,0.9610,0.9771,742.3988
8,00000077,3911,0.8205,0.9951,1287.0885
9,00000091,874,0.9348,0.9771,328.8502


one arc per GPS point on every trip: True


## Request log printed by the server (one line per request)

In [4]:
print("\n".join(buf.getvalue().strip().splitlines()[:6]))

14:26:44 | testclient | GET /health | - | 200 | n=- | acc=- | 1 ms
14:26:44 | testclient | GET /trips | - | 200 | n=- | acc=- | 1 ms
14:26:46 | testclient | POST /predict | 00000000 | 200 | n=2503 | acc=0.968 | 2479 ms
14:26:47 | testclient | POST /predict | 00000000 | 200 | n=2503 | acc=0.968 | 737 ms
14:26:47 | testclient | POST /predict | 00000007 | 200 | n=1354 | acc=0.994 | 438 ms
14:26:48 | testclient | POST /predict | 00000009 | 200 | n=1049 | acc=0.914 | 451 ms


## Ground truth never reaches the UI
* The response contains only `trip_id, n_points, points[point_idx, lon, lat, arc_id], route_coords, accuracy, route_continuity, latency_ms, model_version`.
* The reference route is read only in `api/evaluation.py`, after the prediction exists, and only two aggregate numbers leave that function.
* `tests/test_ui_contract.py` and `tests/test_api.py` lock this response shape; `tests/test_pipeline.py` proves prediction works on a trip folder with **no `.route` file**.

In [5]:
assert set(first) == {"trip_id","n_points","points","route_coords","accuracy","route_continuity","latency_ms","model_version"}
assert set(first["points"][0]) == {"point_idx","lon","lat","arc_id"}
print("response schema OK: no reference arcs or route in the payload")

response schema OK: no reference arcs or route in the payload


## Evaluator UI
The UI (`ui/`) is a static page served by `python -m ui.app`; it calls `/health`, `/trips` and `/predict` and validates every response against the same contract. Start it with `scripts/start_ui.ps1 -Api http://<laptop-1-ip>:8000` and open http://127.0.0.1:8501.